# N108 · 划分DP与最后一段

**目标：** 用最后一段边界枚举前缀最优并计算段成本。

**先修：** N098, N031, N107。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** dp[i]/dp[k][i]；段成本；前缀优化；最少/恰好K段；无效状态。

**配套讲解来源：** [同名逐章意见](../../comment/108_partition_dp.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章处理"把一个数组或字符串切成若干段，每段按某种规则计费，求总费用最优"的问题。三道代表问题分别是：

1. **最大分块和**：把数组切成若干段，每段最长 k，每段的得分是"段内最大值 × 段长"，求所有元素得分之和的最大值。
2. **回文划分 K 段**：把字符串切成恰好 k 段，每段可以改一些字符使它变成回文，改一个字符代价 1，求最小总修改数。
3. **工作日安排**：把 n 个**必须按顺序**做的工作分给 d 天，每天至少一项、工作顺序不能打乱，每天难度是当天工作的最大值，求 d 天难度之和的最小值；分不开就返回 −1。

三个问题各配一个一眼能懂的小例子：`'abc'` 切 2 段（如 `'a'+'bc'`，把 `'bc'` 改成回文要动 1 个字符）最少改 1 个；工作 `[6,5,4,3,2,1]` 分 2 天，最优是 `[6,5,4,3,2] | [1]`，难度和 6+1=7；分块和的完整例子在下面。

先看一个具体到数字的例子（“运行示例”部分用的就是它）：

- 输入 `arr = [1,15,7,9,2,5,10]`，`k = 3`（每段最长 3 个）。
- 输出 `84`。
- 为什么是 84？最优切法是 `[1,15,7] | [9] | [2,5,10]`：第一段最大值 15、长 3，得 45；第二段最大值 9、长 1，得 9；第三段最大值 10、长 3，得 30。合计 45+9+30=84。你可以体会一下套路：把大数塞进尽量长的段里"按次数收费"，最划算。

## 2. 基础知识：先读懂这些词

- **划分 DP**：子问题是"前 i 个元素切完后的最优值"，记作 `dp[i]`（如果还限制段数，就再加一维，`dp[k][i]` = 前 i 个元素恰好切成 k 段的最优值）。它和区间 DP 的区别是：划分 DP 从左往右推进，子问题是一个前缀；区间 DP 的子问题是一段任意区间。
- **最后一段**：本章题眼。任何一个划分，都有唯一确定的"最后一个切点"——最后一段从某个位置开始到末尾结束。我们枚举最后一段的起点 j，那么"前 j 个元素的最优划分"是一个规模更小的同类子问题，两者拼起来就是完整答案。每个合法划分恰好对应一个 j，不会重复也不会遗漏。
- **段成本（cost）**：一段的计费函数，比如"段内最大值 × 段长"、"把这段改成回文要改几个字符"、"这段工作的最大难度"。DP 的转移永远是 `dp[i] = 最优(dp[j] + 最后一段的成本)`。
- **前缀优化**：计算最后一段成本时，常常能边枚举边增量维护。比如"段内最大值"可以在起点往左扩的过程中用一个变量 `maximum` 实时取 max，不需要每段重新扫一遍。
- **无效状态（用 `float('inf')` / `float('-inf')` 标记）**：求最小值时用正无穷表示"到不了的状态"（比如 0 个字符切成 1 段），求最大值时用负无穷。取 min/max 时这些值自动被淘汰，等价于把非法方案排除在候选之外。
- **恰好 d 段 vs 至多 d 段**：工作日安排要求"恰好 d 天且每天非空"，这比"至多 d 段"严格——如果允许空段，你可以用 0 成本的空天凑数，答案会偏小。所以代码里必须保证每段至少一个元素。
- **滚动数组（prev/dp 两行）**：当状态多了一维"段数"时，算第 g 段只需要第 g−1 段的结果，所以只保留两行（`prev` 和 `dp`）轮流滚动，空间从 O(kn) 降到 O(n)。

## 3. 思路推导：从小例子开始

### 主线：最大分块和 `[1,15,7]` 起步

- **Step 1：定状态。** `dp[i]` = 前 i 个元素按规则切分的最大得分，`dp[0]=0`（一个元素都不切，得分 0）。
- **Step 2：枚举最后一段。** 最后一段必然是 `arr[i-length .. i-1]`（长为 length，1 ≤ length ≤ min(k,i)）。它的成本是"这段的最大值 × length"。转移：`dp[i] = max(dp[i-length] + maximum*length)`。
- **Step 3：增量维护最大值。** 注意代码里 length 从 1 往大枚举，每扩一格就 `maximum=max(maximum, arr[i-length])`，把新纳入的元素考虑进来，这样每步 O(1)，不用重复找最大值。
- **Step 4：手算 `a=[1,15,7]`，k=3。**
  - `dp[0]=0`。
  - `dp[1]`：最后一段只能是 `[1]`，最大值 1，得 1×1=1。`dp[1]=1`。
  - `dp[2]`：切 `[1,15]`（15×2=30，dp[0]+30）或 `[15]` 接 dp[1]=1（+15）→ 取 30。`dp[2]=30`。
  - `dp[3]`：三种切法：`[7]` 接 30（37）；`[7,15]`（最大 15，30）接 1（31）；`[1,15,7]` 整段（15×3=45）。取 45。`dp[3]=45`。
- **Step 5：算全长。** “运行示例”部分对 `a=[1,15,7,9,2,5,10]` 逐前缀算出了下面这张表，你可以逐行核对（前 4 行就是上面手算的延续）：

| 前缀长度 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| 最大分块和 k=3 | 0 | 1 | 30 | 45 | 54 | 63 | 72 | 84 |

末尾几个值的来历：dp[4]=dp[3]+9=54（`[9]` 单独一段）；dp[6]=dp[3]+9×3=72（`[9,2,5]` 一段）；dp[7]=dp[4]+10×3=84（`[2,5,10]` 一段）——正是第一节说的最优切法。

### 副线一：回文划分 `'abc'`、k=2

- 先把"任意子串改成回文的代价"预处理成一张表 `cost[l][r]`：比较两端 `s[l]` 和 `s[r]`，不相等代价 1；剥掉两端后里面那段的代价是 `cost[l+1][r-1]`（区间 DP 的老朋友）。`'abc'`：cost['a','b']=1，cost['b','c']=1，cost['a','c']=1（a≠c 得 1，里面只剩 1 个字符代价 0）。
- 再按段数滚动：`prev` 表示"少一段"的最优。切 1 段：`prev=[0, 0, 1, 1]`（前 i 个字符整段改回文的代价）。切 2 段时对 i=3（整串）：枚举最后一段起点 j=1（`'bc'` 代价 1，前面 `'a'` 代价 0）或 j=2（`'c'` 代价 0，前面 `'ab'` 代价 1），都等于 1。答案 1。

### 副线二：工作日安排 `[6,5,4,3,2,1]`、d=2

- 按顺序切 2 刀不减序：`[6]|[5,4,3,2,1]`=6+5=11、`[6,5]|[4,3,2,1]`=6+4=10、`[6,5,4]|[3,2,1]`=6+3=9、`[6,5,4,3]|[2,1]`=6+2=8、`[6,5,4,3,2]|[1]`=6+1=7。最小值 7。注意工作是递减的，所以"越早切，前面那天越难、后面越容易"，最优是让前面尽量长。

## 4. 核心代码：max_sum_after_partitioning

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def max_sum_after_partitioning(arr, k):
    if k <= 0:
        raise ValueError('positive segment limit required')
    dp = [0] + [float('-inf')] * len(arr)
    for i in range(1, len(arr) + 1):
        maximum = float('-inf')
        for length in range(1, min(k, i) + 1):
            maximum = max(maximum, arr[i - length])
            dp[i] = max(dp[i], dp[i - length] + maximum * length)
    return dp[-1]
```

### 逐段读懂代码

### 1. `max_sum_after_partitioning`

```python
def max_sum_after_partitioning(arr, k):
    if k <= 0:
        raise ValueError('positive segment limit required')
    dp = [0] + [float('-inf')] * len(arr)
    for i in range(1, len(arr) + 1):
        maximum = float('-inf')
        for length in range(1, min(k, i) + 1):
            maximum = max(maximum, arr[i - length])
            dp[i] = max(dp[i], dp[i - length] + maximum * length)
    return dp[-1]
```

- `if k<=0: raise`：段长上限必须是正数，0 或负数意味着一段都切不出来，直接报错挡住非法输入。
- `dp=[0]+[float('-inf')]*len(arr)`：dp[0]=0 是唯一合法起点；其余先填负无穷，表示"还没算/到不了"。
- 外层 `i` 是前缀长度；内层 `length` 是最后一段的长度，上限 `min(k,i)` 表示"既不能超过段长限制 k，也不能超过现有元素数 i"。
- 内层那行先 `maximum=max(maximum,arr[i-length])` 把刚并入段的元素纳入最大值统计，再做转移 `dp[i]=max(dp[i], dp[i-length]+maximum*length)`。length 从 1 递增意味着段从右端往左一格一格扩，`maximum` 顺势增量更新，这就是"前缀优化"。
- `return dp[-1]`：整个数组的最优。

### 2. `palindrome_partition_k`

（对应程序见下方分段实现与完整代码。）

- 前三行是边界：k=0 且串为空则 0（不用切也不用改）；k=0 但串非空、k 为负、k 超过字符数（每段至少一个字符）都无解，返回 −1。
- `cost` 表是嵌套的小区间 DP：`cost[l][r]` = 把 `s[l..r]` 改成回文的最少替换数。`(s[l]!=s[r])` 在 Python 里是布尔值，参与加法时 True 当 1、False 当 0。`gap>1` 的判断保证剥掉两端后子区间 `[l+1,r-1]` 至少还有一个字符时才去查表（gap=1 时中间是空的，代价 0）。
- `prev=[0]+[float('inf')]*n`：切 0 段时，只有空串（下标 0）代价为 0，非空串到不了。
- 滚动循环：`groups` 从 1 到 k，`dp[i]` = 前 i 个字符切成 groups 段的最小代价。转移枚举最后一段是 `s[j..i-1]`（代码里查 `cost[j][i-1]`），前面 `j` 个字符切成 groups−1 段用 `prev[j]`。
- `j in range(groups-1,i)`：下界 `groups-1` 保证前 groups−1 段每段至少一个字符（前缀至少要那么长才切得开）；上界 `i`（不含）保证最后一段 `s[j..i-1]` 非空。两头一起堵住"空段"，这正是"恰好 k 段"的要求。
- `prev=dp` 滚到下一轮，最后 `prev[n]` 是整串恰好 k 段的答案。

### 3. `min_difficulty`

```python
def min_difficulty(jobs, d):
    n = len(jobs)
    if d == 0:
        return 0 if not n else -1
    if n < d or d < 0:
        return -1
    prev = [0] + [float('inf')] * n
    for day in range(1, d + 1):
        dp = [float('inf')] * (n + 1)
        for i in range(day, n + 1):
            maximum = float('-inf')
            for j in range(i - 1, day - 2, -1):
                maximum = max(maximum, jobs[j])
                dp[i] = min(dp[i], prev[j] + maximum)
        prev = dp
    return prev[n]
```

- 边界与上一函数同理：d=0 且没工作则 0；工作数少于天数（每天至少一项，塞不下）或 d 为负都返回 −1。
- 滚动结构同款：`prev[j]` = 前 j 个工作安排完前面若干天的最小难度和，第 `day` 天接管 `jobs[j..i-1]`。
- 内层 `j` 从 `i-1` **倒着**枚举到 `day-1`（`range(i-1,day-2,-1)` 的终点 `day-2` 是排他的，所以最后一个 j 是 `day-1`）。倒序的好处：j 每减 1，最后一段就向左扩一格，`maximum=max(maximum,jobs[j])` 顺手把新工作纳入当天最大值统计，O(1) 更新——和第一个函数的"前缀优化"是同一个技巧，只是方向反过来。
- 下界 j ≥ day−1 的含义：前面 day−1 天每天至少一项工作，所以前缀至少要有 day−1 项，`prev[day-1]` 才是可能非无穷的最早位置。

## 5. 分段实现：按顺序运行

**本章接口：** `max_sum_after_partitioning(arr, k)`、`palindrome_partition_k(s, k)`、`min_difficulty(jobs, d)`

### max_sum_after_partitioning

In [1]:
def max_sum_after_partitioning(arr, k):
    if k <= 0:
        raise ValueError('positive segment limit required')
    dp = [0] + [float('-inf')] * len(arr)
    for i in range(1, len(arr) + 1):
        maximum = float('-inf')
        for length in range(1, min(k, i) + 1):
            maximum = max(maximum, arr[i - length])
            dp[i] = max(dp[i], dp[i - length] + maximum * length)
    return dp[-1]

### palindrome_partition_k

In [2]:
def palindrome_partition_k(s, k):
    n = len(s)
    if k == 0:
        return 0 if not n else -1
    if k < 0 or k > n:
        return -1
    cost = [[0] * n for _ in range(n)]
    for gap in range(1, n):
        for l in range(n - gap):
            r = l + gap
            cost[l][r] = (cost[l + 1][r - 1] if gap > 1 else 0) + (s[l] != s[r])
    prev = [0] + [float('inf')] * n
    for groups in range(1, k + 1):
        dp = [float('inf')] * (n + 1)
        for i in range(groups, n + 1):
            dp[i] = min((prev[j] + cost[j][i - 1] for j in range(groups - 1, i)))
        prev = dp
    return prev[n]

### min_difficulty

In [3]:
def min_difficulty(jobs, d):
    n = len(jobs)
    if d == 0:
        return 0 if not n else -1
    if n < d or d < 0:
        return -1
    prev = [0] + [float('inf')] * n
    for day in range(1, d + 1):
        dp = [float('inf')] * (n + 1)
        for i in range(day, n + 1):
            maximum = float('-inf')
            for j in range(i - 1, day - 2, -1):
                maximum = max(maximum, jobs[j])
                dp[i] = min(dp[i], prev[j] + maximum)
        prev = dp
    return prev[n]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,15,7,9,2,5,10]; show_table(['前缀长度','最大分块和，k=3'],[(i,max_sum_after_partitioning(a[:i],3)) for i in range(len(a)+1)])

前缀长度,最大分块和，k=3
0,0
1,1
2,30
3,45
4,54
5,63
6,72
7,84


## 7. 正确性、适用条件与复杂度

每个合法划分都有唯一最后切点j；前缀最优加最后段代价枚举了全部划分。段数g的前缀至少长g，限制j≥g−1保证每段非空。回文修复成本等于相对位置不相等的对数，各对独立。

**代价：** 最大分块和O(nk)时间O(n)空间；回文划分O(n²+kn²)时间O(n²)空间；工作安排O(dn²)时间O(n)辅助。固定段数不合法时返回−1。

### 展开理解

**为什么是对的？** 我们以"最后一段"为锚点论证。任何一种合法划分，它的最后一段是确定的，起点 j 唯一；把最后一段砍掉，剩下的是前 j 个元素的一个合法划分（段数少一、规则相同），反之亦然。所以"完整划分的最优值"就等于"某个更小前缀的最优值 + 最后一段成本"，对 j 取 min/max 把所有划分一网打尽，既不会算重（一个划分只有一个 j），也不会算漏（每个 j 都被枚举了）。三个函数里对 j 范围的限制（`length ≤ k`、`j ≥ groups-1`、`j ≥ day-1`）保证枚举出的每一段都满足"非空"和"长度上限"这些题目约束；负无穷/正无穷初始化则保证只有真正可达的状态才会胜出。回文成本表的正确性来自一个简单观察：要让 `s[l..r]` 成为回文，位置 l 必须和位置 r 配对相等（不等就改一个，代价 1），里面 `s[l+1..r-1]` 独立地自成一个子问题，每一对位置互不干扰，所以代价就是"不相等的对数"。

**复杂度**：最大分块和是 O(n·k) 时间、O(n) 空间——n=10⁵、k=100 时约一千万次操作，一瞬间跑完。回文划分是预处理 O(n²) 加主 DP O(k·n²)（外层段数 k 轮、每轮对每个 i 枚举 j），空间 O(n²) 存 cost 表；n=100 时 cost 表一万个格子，毫无压力。工作安排是 O(d·n²) 时间、O(n) 空间。规模再大就要考虑单调队列之类的优化，但那是后话。

## 8. 单元测试：每个用例在检查什么

- `assert max_sum_after_partitioning([1,15,7,9,2,5,10],3)==84`：正常值测试，即第一、三节手算的例子，检查"长段放大值"的策略与转移实现。
- `assert min_difficulty([6,5,4,3,2,1],2)==7 and min_difficulty([1,2],3)==-1`：前半段是正常值（第三节手算的 7）；后半段是边界测试——只有 2 项工作却要排 3 天，每天非空塞不下，必须返回 −1，专测 `n<d` 那行守卫。
- `assert palindrome_partition_k('abc',2)==1 and palindrome_partition_k('abc',3)==0`：`'abc'` 切 2 段（如 `'a'+'bc'`，`'bc'` 要改 1 个）得 1；切 3 段每个单字符天然回文，得 0。后者同时验证了"单字符段的回文代价为 0"这条边界。
- 后半部分是与暴力解的全量对拍：`partitions(a,k)` 用 `combinations` 枚举所有 k−1 个切点位置，生成全部 k 段划分；然后对 3⁵=243 个长度 5 的数组/字符串，逐一断言三个函数与"直接按定义算"的结果一致——`min_difficulty` 对每段取 max 求和再取 min；回文对每段数首尾不相等的对数；分块和对任意段数（1 到 5）、段长不超过 limit 的所有划分取最大。特别值得注意：分块和的暴力允许任意段数，说明该函数的契约是"只限段长、不限段数"，你的理解要和这个契约对齐。

In [5]:
assert max_sum_after_partitioning([1, 15, 7, 9, 2, 5, 10], 3) == 84

assert min_difficulty([6, 5, 4, 3, 2, 1], 2) == 7 and min_difficulty([1, 2], 3) == -1

assert palindrome_partition_k('abc', 2) == 1 and palindrome_partition_k('abc', 3) == 0

from itertools import combinations, product

def partitions(a, k):
    for cuts in combinations(range(1, len(a)), k - 1):
        ends = (0,) + cuts + (len(a),)
        yield [a[ends[i]:ends[i + 1]] for i in range(k)]

for a in product(range(3), repeat=5):
    s = ''.join(map(str, a))
    for k in range(1, 6):
        ps = list(partitions(a, k))
        sp = list(partitions(s, k))
        assert min_difficulty(a, k) == min((sum((max(x) for x in p)) for p in ps))
        assert palindrome_partition_k(s, k) == min((sum((sum((x[i] != x[-1 - i] for i in range(len(x) // 2))) for x in p)) for p in sp))
    for limit in (1, 2, 3):
        want = max((sum((max(x) * len(x) for x in p)) for k in range(1, 6) for p in partitions(a, k) if all((len(x) <= limit for x in p))))
        assert max_sum_after_partitioning(a, limit) == want

print('N108: 所有本章断言通过')

N108: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 返回划分边界。

**练习 2：** 比较恰好d段与至多d段。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（返回划分边界）**：提示——现在函数只返回最优值，你要让它把切点也吐出来。两条路任选：一是照着 dp 表"复路"，从 `dp[n]` 出发，找到是哪个 `length`/`j` 达到了这个值，一路倒推回 dp[0]，把每段起点记下来再反转；二是转移时多存一个 `choice[i]` 数组，直接记录"最后一段长度/起点"，最后顺着走一遍。手算例子用 `[1,15,7,9]`、k=3（最优是整段前三个加 `[9]`），验证你输出的边界确实能拼出 54。
- **练习 2（恰好 d 段 vs 至多 d 段）**：提示——找一组让两者答案不同的输入。比如工作安排里若允许"至多 d 天"，某些天可以空着，难度和可能更小。拿 `jobs=[9,1,1]`、d=2 做手算：恰好 2 天是 `[9]|[1,1]`=9+1=10；如果允许 1 天是 9。再想想字符串划分里"恰好 k 段"改成"至多 k 段"答案会不会变（提示：单字符段回文代价为 0，多切往往不亏也不赚）。最后说明本章代码里 `j ≥ groups-1`、`j ≥ day-1` 这些下界正是"恰好"二字的实现，去掉它们会得到什么行为。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def max_sum_after_partitioning(arr, k):
    if k <= 0:
        raise ValueError('positive segment limit required')
    dp = [0] + [float('-inf')] * len(arr)
    for i in range(1, len(arr) + 1):
        maximum = float('-inf')
        for length in range(1, min(k, i) + 1):
            maximum = max(maximum, arr[i - length])
            dp[i] = max(dp[i], dp[i - length] + maximum * length)
    return dp[-1]

def palindrome_partition_k(s, k):
    n = len(s)
    if k == 0:
        return 0 if not n else -1
    if k < 0 or k > n:
        return -1
    cost = [[0] * n for _ in range(n)]
    for gap in range(1, n):
        for l in range(n - gap):
            r = l + gap
            cost[l][r] = (cost[l + 1][r - 1] if gap > 1 else 0) + (s[l] != s[r])
    prev = [0] + [float('inf')] * n
    for groups in range(1, k + 1):
        dp = [float('inf')] * (n + 1)
        for i in range(groups, n + 1):
            dp[i] = min((prev[j] + cost[j][i - 1] for j in range(groups - 1, i)))
        prev = dp
    return prev[n]

def min_difficulty(jobs, d):
    n = len(jobs)
    if d == 0:
        return 0 if not n else -1
    if n < d or d < 0:
        return -1
    prev = [0] + [float('inf')] * n
    for day in range(1, d + 1):
        dp = [float('inf')] * (n + 1)
        for i in range(day, n + 1):
            maximum = float('-inf')
            for j in range(i - 1, day - 2, -1):
                maximum = max(maximum, jobs[j])
                dp[i] = min(dp[i], prev[j] + maximum)
        prev = dp
    return prev[n]

# 示例与回归测试
assert max_sum_after_partitioning([1, 15, 7, 9, 2, 5, 10], 3) == 84

assert min_difficulty([6, 5, 4, 3, 2, 1], 2) == 7 and min_difficulty([1, 2], 3) == -1

assert palindrome_partition_k('abc', 2) == 1 and palindrome_partition_k('abc', 3) == 0

from itertools import combinations, product

def partitions(a, k):
    for cuts in combinations(range(1, len(a)), k - 1):
        ends = (0,) + cuts + (len(a),)
        yield [a[ends[i]:ends[i + 1]] for i in range(k)]

for a in product(range(3), repeat=5):
    s = ''.join(map(str, a))
    for k in range(1, 6):
        ps = list(partitions(a, k))
        sp = list(partitions(s, k))
        assert min_difficulty(a, k) == min((sum((max(x) for x in p)) for p in ps))
        assert palindrome_partition_k(s, k) == min((sum((sum((x[i] != x[-1 - i] for i in range(len(x) // 2))) for x in p)) for p in sp))
    for limit in (1, 2, 3):
        want = max((sum((max(x) * len(x) for x in p)) for k in range(1, 6) for p in partitions(a, k) if all((len(x) <= limit for x in p))))
        assert max_sum_after_partitioning(a, limit) == want

print('N108: 所有本章断言通过')

N108: 所有本章断言通过


## 11. 代表题与迁移

- **1043. Partition Array for Maximum Sum**：练"最后一段长度枚举 + 增量维护段内最大值"，对应 `max_sum_after_partitioning`。
- **1278. Palindrome Partitioning III**：练"先用区间 DP 预处理段成本，再按段数滚动求恰好 k 段"的两段式结构，对应 `palindrome_partition_k`。
- **1335. Minimum Difficulty of a Job Schedule**：练"按顺序划分 + 段成本是区间最大值 + 恰好 d 段每天非空"，对应 `min_difficulty`，倒序枚举 j 增量维护 maximum 是它的招牌技巧。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。